# Day 2.5 — Citations and Abstention
Notebook 04 ended with a generator that had no way to say "the documents do not cover
this". We now force a structured answer and then **check it ourselves**.

```text
evidence sufficient   -> answer + citations that we verify
evidence insufficient -> abstain, no citations
```


## Before you begin

### Learning outcomes

- Require a structured answer whose citations name the chunk ids we supplied.
- Validate those citations in application code and drop the ones we never supplied.

Architecture reference: [D07](../diagrams/source/day_02.md).

### Expected observation

A valid answer keeps its citation and reports `grounded=True`; an answer carrying an
invented chunk id has it removed and reports `grounded=False`; an unanswerable question
abstains with zero citations.

### Modes

Runs offline by default. With a key in `.env` the same cells call the live model.

## Concept briefing

## Citations and abstention

A citation should identify evidence the application actually supplied. Asking the model
to "always cite sources" is insufficient; the host must verify that every returned chunk
identifier belongs to a chunk it retrieved, and drop the ones that do not. For the same
reason the application, not the model, decides whether an answer is grounded: a field in
which the model declares its own answer trustworthy proves nothing.

When evidence is missing, abstention is a successful safety behavior. It tells downstream
users that another information source or human decision is required. A well formed
abstention carries no citations at all.


In [ ]:
# --- Course setup: run this cell first -----------------------------------------
# 1) Locate this day's folder so we can import from src/ and read data/ no matter
#    where Jupyter, VS Code, or Colab started. Every file path below goes through
#    PROJECT_ROOT, never through the current working directory.
import os, sys
from pathlib import Path

def find_project_root(marker="src/knowledge_agent"):
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        for candidate in [folder, *folder.glob("day_*")]:
            if (candidate / marker).exists():
                return candidate
    raise FileNotFoundError(
        "Course folder not found. On Google Colab run the 'Colab bootstrap' cell at the top "
        "of the day notebook first; locally, start Jupyter inside the repository folder."
    )

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

# 2) Load the API key from the .env file at the repository root (Day 1.1 shows how to
#    create it). If no key is present we stay in deterministic MOCK mode: every cell
#    still runs, answers are fixed strings, and no credit is spent.
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))
LIVE = bool(os.getenv("OPENROUTER_API_KEY"))

print("Project root :", PROJECT_ROOT)
print("Mode         :", "LIVE (OpenRouter)" if LIVE else "MOCK (no OPENROUTER_API_KEY found)")

# 3) Same retrieval stack as notebook 04, plus the citation tools.
import json

from knowledge_agent.assistant import validate_citations
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import load_embedder
from knowledge_agent.generation import (
    MockGroundedGenerator,
    OpenRouterGroundedGenerator,
    strict_json_schema,
)
from knowledge_agent.retrieval import VectorIndex
from knowledge_agent.schemas import Citation, GroundedAnswer, ModelAnswer

chunks = load_markdown_corpus(PROJECT_ROOT / "data" / "corpus")
embedder, embedder_label = load_embedder()
index = VectorIndex(embedder)
index.add(chunks)

generator = MockGroundedGenerator()
if LIVE:
    try:
        generator = OpenRouterGroundedGenerator()
    except Exception as exc:
        print("Live generator unavailable, staying offline:", exc)

print("Generator    :", type(generator).__name__)

## Step 1 — Describe the answer we will accept

Free text cannot be checked. We define the answer as a small object - text, citations,
abstained - and send its JSON schema with the request so the provider must return that
shape. `strict_json_schema` closes the schema first (see Step 2).

In [ ]:
schema = strict_json_schema(ModelAnswer)
print(json.dumps(schema, indent=1))

## Step 2 — Why the schema needs post-processing

Pydantic writes a permissive schema. Strict structured-output modes additionally require
`additionalProperties: false`, every property listed in `required`, and no `default`
values - otherwise the request is rejected with HTTP 400. `strict_json_schema` walks the
schema (including `$defs`) and adds exactly that.

Notice which field is **absent**: `grounded`. We never ask a model to certify its own
answer; the application decides that in Step 4.

In [ ]:
print("top level closed to extra keys :", schema["additionalProperties"] is False)
print("every property required        :", sorted(schema["properties"]) == schema["required"])
print("nested Citation closed         :", schema["$defs"]["Citation"]["additionalProperties"] is False)
print("model asked for 'grounded'?    :", "grounded" in schema["properties"])
print()
print("Raw Pydantic schema for comparison:")
print(json.dumps(ModelAnswer.model_json_schema()["properties"]["citations"], indent=1))

## Step 3 — Answer a question the corpus covers

Retrieve, generate, and print the structured result. The citation must name one of the
chunk ids that appeared in the evidence context.

In [ ]:
QUESTION = "How long are battery fault-event records retained?"
retrieved = index.search(QUESTION, top_k=3)
supplied_ids = [item.chunk.chunk_id for item in retrieved]

try:
    answer = generator.generate(QUESTION, retrieved)
except Exception as exc:
    print("Live call failed, using the offline generator:", exc)
    answer = MockGroundedGenerator().generate(QUESTION, retrieved)

print("supplied to the model :", supplied_ids)
print("abstained             :", answer.abstained)
print("citations returned    :", [citation.chunk_id for citation in answer.citations])
print("answer                :", answer.answer)

## Step 4 — Validate the citations in application code

`validate_citations` keeps only citations whose `chunk_id` we actually retrieved *and*
whose source and section match that chunk. Whatever survives sets `grounded` - a field the
application owns.

In [ ]:
validated = validate_citations(answer, retrieved)

print("kept citations    :", [citation.chunk_id for citation in validated.citations])
print("dropped citations :", [citation.chunk_id for citation in validated.dropped_citations])
print("grounded          :", validated.grounded)
print()
print("grounded=True means: not an abstention, at least one citation survived, and nothing")
print("had to be dropped. It is computed from our own retrieval log, so it cannot be faked.")

## Step 5 — Break it: an invented citation

Models do produce citations that were never supplied - copied from a previous answer, or
simply plausible-looking. Build that answer by hand and watch the check reject it.

In [ ]:
tampered = GroundedAnswer(
    answer="Fault records are retained for one year.",
    citations=[
        Citation(source="battery_safety.md", section="Data retention", chunk_id="battery_safety:data-retention"),
        Citation(source="battery_safety.md", section="Appendix C", chunk_id="battery_safety:appendix-c"),  # never existed
    ],
    abstained=False,
)

checked = validate_citations(tampered, retrieved)
print("kept    :", [citation.chunk_id for citation in checked.citations])
print("dropped :", [citation.chunk_id for citation in checked.dropped_citations])
print("grounded:", checked.grounded)
print()
print("The real citation survives; the invented one is removed and the answer is flagged")
print("as not grounded, so a caller can refuse to display it.")

## Step 6 — Abstain when the evidence is missing

The unanswerable question from notebook 04 now has a defined outcome: `abstained=True`
with zero citations. A well formed abstention is a *successful* result, not an error.

In [ ]:
UNANSWERABLE = "What is the purchase price of the battery system?"
missing_evidence = index.search(UNANSWERABLE, top_k=3)

try:
    refusal = generator.generate(UNANSWERABLE, missing_evidence)
except Exception as exc:
    print("Live call failed, using the offline generator:", exc)
    refusal = MockGroundedGenerator().generate(UNANSWERABLE, missing_evidence)

refusal = validate_citations(refusal, missing_evidence)
print("retrieved anyway :", [item.chunk.chunk_id for item in missing_evidence])
print("abstained        :", refusal.abstained)
print("citations        :", refusal.citations)
print("grounded         :", refusal.grounded, "(an abstention is grounded when it cites nothing)")
print("answer           :", refusal.answer)

## Step 7 — The same checks as a score

`evaluate_answers` runs a whole pipeline over golden cases and records
`citation_provenance_ok`, which is exactly the `grounded` flag from Step 4. Two cases are
enough to see the column; notebook 06 runs all ten.

In [ ]:
from knowledge_agent.assistant import KnowledgeAssistant
from knowledge_agent.evaluation import evaluate_answers, render_table
from knowledge_agent.schemas import GoldenCase

assistant = KnowledgeAssistant(index, MockGroundedGenerator(), top_k=3)   # offline: no credit spent
two_cases = [
    GoldenCase(
        id="demo-answerable",
        question=QUESTION,
        answerable=True,
        expected_source="battery_safety.md",
        expected_section="Data retention",
        essential_terms=["one year"],
    ),
    GoldenCase(
        id="demo-unanswerable",
        question=UNANSWERABLE,
        answerable=False,
        expected_source=None,
        expected_section=None,
        essential_terms=[],
    ),
]
records = evaluate_answers(assistant, two_cases)
print(render_table(records, ["id", "abstained", "abstention_correct", "citation_correct",
                             "citation_provenance_ok", "dropped_citations", "essential_term_coverage"]))

### Try it yourself

Invent a question the corpus cannot answer - not about price. Predict whether the offline
generator abstains, then check.

In [ ]:
# --- Worked solution ---
from knowledge_agent.generation import distinctive_matches

MY_QUESTIONS = [
    "What is the wifi password for the campus network?",
    "How many parking spaces does the campus have?",
    "Who manufactured the battery cells?",
]
for question in MY_QUESTIONS:
    evidence = index.search(question, top_k=3)
    result = validate_citations(MockGroundedGenerator().generate(question, evidence), evidence)
    matched = {chunk_id: words for chunk_id, words in distinctive_matches(question, evidence).items() if words}
    print(question)
    print(f"   abstained={result.abstained}  citations={len(result.citations)}  matched words={matched}")

print()
print("The first two abstain because no retrieved chunk contains a specific word from the")
print("question - there is no list of forbidden topics anywhere in the code.")
print("The third is fooled: the battery section contains the word 'cell', so the lexical")
print("rule believes it has evidence about who manufactured them. A real model reads the")
print("passage and sees no manufacturer, which is exactly what LIVE mode is for.")

### Checkpoint

**1. The model returned `"grounded": true`. Why do we ignore that field?**

<details><summary>Show answer</summary>

Because it is generated by the same process that produced the answer, so it adds no
independent information - a model that invents a citation will happily also claim to be
grounded. Our `grounded` flag is computed from the retrieval log we kept ourselves, which
is why `ModelAnswer` does not even offer the field to the model.

</details>

**2. An abstention arrives with two citations attached. Is that acceptable?**

<details><summary>Show answer</summary>

No, and `validate_citations` marks it `grounded=False`. Abstention means "the supplied
evidence does not support an answer"; attaching sources to that claim is self
contradictory and misleads whoever reads the result. The contract is: an answer cites, an
abstention does not.

</details>

### Recap

- **Limitation we saw:** a free-text answer cannot be checked, and citations can name
  chunks that were never retrieved.
- **Layer we added:** a strict answer schema plus application-side citation validation
  that sets `grounded` and records what was dropped.
- **Evidence it worked:** the invented `battery_safety:appendix-c` citation is removed and
  the answer is flagged, while the abstention returns zero citations.